# ML-07 - Baseline Action Score and Top-10 Review

Lane: **Refresh / Content Opportunity Scoring**.

Plain-English rule: put pages into the first review queue when they are visible enough to matter, sit in a reachable search-results position, have low CTR, and are stale enough that a content or metadata refresh is plausible. I deliberately do **not** use `trend_direction`, `trend_pct`, or the 30-day trend inputs as rule features because the starter label is derived from those fields.

This notebook writes `work/outputs/baseline_action_score.csv`. The CSV is ignored by git by design; the notebook and JSON receipt are the commit-worthy artifacts.

## 1. My rule and its reason code

Two signal checks first:

1. **Staleness behind refresh flags:** if pages have not been updated recently, are they more often declining?
2. **CTR-vs-position behind CTR-fix logic:** if a page has impressions but weak CTR in useful positions, does that mark review-worthy pages?

Verdict choices are one word: CONFIRMED, OPPOSITE, MIXED, or FALSE.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data/raw/content_refresh_anonymized.csv").exists():
    ROOT = Path.cwd().parents[1]

DATA_PATH = ROOT / "data/raw/content_refresh_anonymized.csv"
OUTPUT_DIR = ROOT / "work/outputs"
QUEUE_PATH = OUTPUT_DIR / "baseline_action_score.csv"
METRICS_PATH = OUTPUT_DIR / "baseline_action_score_metrics.json"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)

print(f"rows n={len(df):,}")
print(f"declining proxy base rate={df['is_declining_label'].mean():.3f} (n={int(df['is_declining_label'].sum()):,})")

rows n=30,000
declining proxy base rate=0.542 (n=16,262)


In [2]:
freshness_table = (
    df.groupby("freshness_tier", dropna=False)
    .agg(
        n=("content_id", "size"),
        declining_n=("is_declining_label", "sum"),
        decline_rate=("is_declining_label", "mean"),
        median_impressions=("impressions_90d", "median"),
    )
    .reindex(["0-30", "31-90", "91-180", "181+", "never"])
    .dropna(how="all")
)

print("Signal 1: staleness behind refresh flags")
print(freshness_table.to_string(float_format=lambda x: f"{x:.3f}"))
print("VERDICT: MIXED")
print(
    "Reason: the 91-180 day bucket has a higher decline rate than recently updated pages, "
    "but the tiny 181+ bucket goes the other way, so staleness is useful only with other filters."
)

Signal 1: staleness behind refresh flags
                       n  declining_n  decline_rate  median_impressions
freshness_tier                                                         
0-30           20480.000    10473.000         0.511             470.000
31-90            175.000      103.000         0.589             510.000
91-180          9171.000     5604.000         0.611            1692.000
181+             174.000       82.000         0.471              15.500
VERDICT: MIXED
Reason: the 91-180 day bucket has a higher decline rate than recently updated pages, but the tiny 181+ bucket goes the other way, so staleness is useful only with other filters.


In [3]:
valid_position = df[df["avg_position"] > 0].copy()
valid_position["position_bucket"] = pd.cut(
    valid_position["avg_position"],
    bins=[0, 3, 10, 20, 50, np.inf],
    labels=["top_3", "page_1_4_10", "striking_11_20", "page_3_5_21_50", "deep_50_plus"],
    include_lowest=True,
)
valid_position["low_ctr"] = (valid_position["ctr"] < 0.5).astype(int)

ctr_position_table = (
    valid_position.groupby("position_bucket", observed=True)
    .agg(
        n=("content_id", "size"),
        low_ctr_n=("low_ctr", "sum"),
        low_ctr_rate=("low_ctr", "mean"),
        median_ctr=("ctr", "median"),
        decline_rate=("is_declining_label", "mean"),
        median_impressions=("impressions_90d", "median"),
    )
)

print("Signal 2: CTR-vs-position behind CTR-fix logic")
print(ctr_position_table.to_string(float_format=lambda x: f"{x:.3f}"))
print("VERDICT: CONFIRMED")
print(
    "Reason: low CTR is common in reachable positions, and the 4-20 position range has "
    "above-base decline rates while still having enough impressions to justify review."
)

Signal 2: CTR-vs-position behind CTR-fix logic
                     n  low_ctr_n  low_ctr_rate  median_ctr  decline_rate  median_impressions
position_bucket                                                                              
top_3             1141        903         0.791       0.000         0.498              74.000
page_1_4_10      11842       9433         0.797       0.160         0.569            1184.000
striking_11_20    7273       6222         0.855       0.100         0.610             870.000
page_3_5_21_50    7225       6735         0.932       0.030         0.562             807.000
deep_50_plus      1314       1259         0.958       0.000         0.343             219.500
VERDICT: CONFIRMED
Reason: low CTR is common in reachable positions, and the 4-20 position range has above-base decline rates while still having enough impressions to justify review.


## 2. Build the ranked queue

Rule inputs:

- `impressions_90d` between 300 and 30,000: visible enough for action, but not the ultra-large pages that dominated the first draft.
- `avg_position` from >3 to <=20: reachable ranking positions where CTR/snippet work can matter.
- `ctr < 0.5`: low CTR as a percent, not a fraction.
- `days_since_last_update >= 91`: stale enough for refresh review.
- `age_tier` in `31-90` or `91-180`: newer/mid-age pages had higher observed decline rates in this slice.

There is exactly one reason code per row, chosen by priority.

In [4]:
work = df.copy()

work["visible_band"] = ((work["impressions_90d"] >= 300) & (work["impressions_90d"] < 30000)).astype(int)
work["reachable_position"] = ((work["avg_position"] > 3) & (work["avg_position"] <= 20)).astype(int)
work["low_ctr_flag"] = (work["ctr"] < 0.5).astype(int)
work["stale_flag"] = (work["days_since_last_update"] >= 91).astype(int)
work["fresh_age_flag"] = work["age_tier"].isin(["31-90", "91-180"]).astype(int)

work["baseline_action_score"] = (
    work["visible_band"]
    * work["reachable_position"]
    * np.log1p(work["impressions_90d"])
    * (
        1
        + 0.45 * work["stale_flag"]
        + 0.35 * work["low_ctr_flag"]
        + 0.25 * work["fresh_age_flag"]
    )
)

conditions = [
    (work["visible_band"].eq(1) & work["reachable_position"].eq(1) & work["stale_flag"].eq(1) & work["low_ctr_flag"].eq(1)),
    (work["visible_band"].eq(1) & work["reachable_position"].eq(1) & work["low_ctr_flag"].eq(1)),
    (work["visible_band"].eq(1) & work["reachable_position"].eq(1) & work["stale_flag"].eq(1)),
    (work["visible_band"].eq(1) & work["reachable_position"].eq(1)),
]
reason_codes = [
    "stale_visible_low_ctr",
    "visible_low_ctr_position",
    "stale_visible_position",
    "visible_position_opportunity",
]
action_labels = [
    "refresh_title_meta_and_content",
    "review_title_meta_ctr",
    "refresh_content",
    "review_search_snippet",
]

work["reason_code"] = np.select(conditions, reason_codes, default="not_in_queue")
work["action_label"] = np.select(
    [work["reason_code"].eq(code) for code in reason_codes],
    action_labels,
    default="monitor",
)

queue = work.sort_values(["baseline_action_score", "impressions_90d"], ascending=[False, False]).copy()
queue["baseline_rank"] = np.arange(1, len(queue) + 1)

queue_columns = [
    "baseline_rank",
    "content_id",
    "client_id",
    "baseline_action_score",
    "reason_code",
    "action_label",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "days_since_last_update",
    "age_tier",
    "freshness_tier",
    "content_type",
    "is_declining_label",
]

queue[queue_columns].to_csv(QUEUE_PATH, index=False)

metrics = {
    "rows": int(len(queue)),
    "base_decline_rate": float(queue["is_declining_label"].mean()),
    "precision_at_10": float(queue.head(10)["is_declining_label"].mean()),
    "precision_at_20": float(queue.head(20)["is_declining_label"].mean()),
    "precision_at_50": float(queue.head(50)["is_declining_label"].mean()),
    "precision_at_100": float(queue.head(100)["is_declining_label"].mean()),
    "queue_path": "work/outputs/baseline_action_score.csv",
    "no_label_derived_inputs": [
        "trend_direction",
        "trend_pct",
        "impressions_last_30d",
        "clicks_last_30d",
        "sessions_last_30d",
        "impressions_prev_30d",
        "clicks_prev_30d",
        "sessions_prev_30d",
    ],
}
METRICS_PATH.write_text(json.dumps(metrics, indent=2), encoding="utf-8")

print(f"Wrote work/outputs/baseline_action_score.csv with n={len(queue):,}")
print(json.dumps(metrics, indent=2))
print(queue[queue_columns].head(10).to_string(index=False))

Wrote work/outputs/baseline_action_score.csv with n=30,000
{
  "rows": 30000,
  "base_decline_rate": 0.5420666666666667,
  "precision_at_10": 0.7,
  "precision_at_20": 0.6,
  "precision_at_50": 0.68,
  "precision_at_100": 0.71,
  "queue_path": "work/outputs/baseline_action_score.csv",
  "no_label_derived_inputs": [
    "trend_direction",
    "trend_pct",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d"
  ]
}
 baseline_rank           content_id         client_id  baseline_action_score           reason_code                   action_label  impressions_90d  clicks_90d  ctr  avg_position  days_since_last_update age_tier freshness_tier    content_type  is_declining_label
             1 content_83163890c43a client_19581e27de              21.121222 stale_visible_low_ctr refresh_title_meta_and_content            29822         130 0.44           3.1                     104   91-180         91-18

## 3. Top-10 review

Each line names the action, why the row is there, and what would make the recommendation wrong.

In [5]:
def wrong_if(row):
    parts = []
    if row["is_declining_label"] == 0:
        parts.append("starter proxy is not declining")
    if row["ctr"] >= 0.5:
        parts.append("CTR is not actually low")
    if row["avg_position"] <= 3:
        parts.append("already top-3, so CTR work may not be the bottleneck")
    if row["impressions_90d"] >= 30000:
        parts.append("too large for the visible band")
    if row["days_since_last_update"] < 91:
        parts.append("recently updated")
    if not parts:
        parts.append("the decline is seasonal, query mix shifted, or the page is intentionally low-CTR")
    return "; ".join(parts)

top10_review = queue.head(10).copy()
top10_review["why_its_there"] = top10_review.apply(
    lambda r: (
        f"{r['reason_code']}: {int(r['impressions_90d']):,} impressions, "
        f"position {r['avg_position']:.1f}, CTR {r['ctr']:.2f}%, "
        f"{int(r['days_since_last_update'])} days since update"
    ),
    axis=1,
)
top10_review["what_would_make_it_wrong"] = top10_review.apply(wrong_if, axis=1)
top10_review["observed_proxy"] = top10_review["trend_direction"]

review_columns = [
    "baseline_rank",
    "content_id",
    "action_label",
    "why_its_there",
    "what_would_make_it_wrong",
    "observed_proxy",
]

print(top10_review[review_columns].to_string(index=False))

 baseline_rank           content_id                   action_label                                                                              why_its_there                                                         what_would_make_it_wrong observed_proxy
             1 content_83163890c43a refresh_title_meta_and_content  stale_visible_low_ctr: 29,822 impressions, position 3.1, CTR 0.44%, 104 days since update the decline is seasonal, query mix shifted, or the page is intentionally low-CTR           down
             2 content_e859812ce999 refresh_title_meta_and_content  stale_visible_low_ctr: 29,760 impressions, position 6.1, CTR 0.05%, 104 days since update the decline is seasonal, query mix shifted, or the page is intentionally low-CTR           down
             3 content_9dff6e7cbb0b refresh_title_meta_and_content  stale_visible_low_ctr: 29,543 impressions, position 5.2, CTR 0.04%, 104 days since update the decline is seasonal, query mix shifted, or the page is intentionally low-CTR

## 4. Weak picks + leakage check

Weak picks are not failures. They show where the rule needs a human reviewer and what the Week-5 model should learn to beat.

In [6]:
weak_picks = top10_review[top10_review["is_declining_label"].eq(0)][
    ["baseline_rank", "content_id", "reason_code", "action_label", "trend_direction", "what_would_make_it_wrong"]
]

print("Weak top-10 picks:")
if weak_picks.empty:
    print("None in the top 10; review top 20 before trusting the rule too much.")
else:
    print(weak_picks.to_string(index=False))

used_rule_inputs = {
    "impressions_90d",
    "avg_position",
    "ctr",
    "days_since_last_update",
    "age_tier",
}
forbidden_inputs = {
    "trend_direction",
    "trend_pct",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
}
leaked = sorted(used_rule_inputs & forbidden_inputs)

print(f"Rule inputs: {sorted(used_rule_inputs)}")
print(f"Forbidden/label-derived inputs used: {leaked}")
assert not leaked
assert QUEUE_PATH.exists()
assert METRICS_PATH.exists()

Weak top-10 picks:
 baseline_rank           content_id           reason_code                   action_label trend_direction       what_would_make_it_wrong
             7 content_8e300c01232e stale_visible_low_ctr refresh_title_meta_and_content          stable starter proxy is not declining
             9 content_6e1955f4f85a stale_visible_low_ctr refresh_title_meta_and_content              up starter proxy is not declining
            10 content_bc5ec602d0b6 stale_visible_low_ctr refresh_title_meta_and_content          stable starter proxy is not declining
Rule inputs: ['age_tier', 'avg_position', 'ctr', 'days_since_last_update', 'impressions_90d']
Forbidden/label-derived inputs used: []


## 5. Self-check

- tick - Two signal verdicts with visible bucket tables and `n`
- tick - At least one FlyRank flag-linked signal checked
- tick - One transparent rule with score, one reason code, and action label
- tick - Ranked queue written from this notebook
- tick - Top-10 reviewed with what would make each pick wrong
- tick - No future-window or label-derived inputs in the rule
- tick - No private names, URLs, raw queries, or datasets committed